# Notebook 4 — Feedback Analysis & Continuous Improvement
## Digital Skills Programme · Q1 2026 · Geneva  
### Text Mining · TF-IDF · LDA Topic Modelling

---

### Purpose
This notebook answers the **continuous improvement** question of the impact measurement exercise:

| # | Core question |
|---|---------------|
| Q8 | Which **feedback aspects** can we use to improve the programme? |

### Why qualitative feedback analysis matters
Numeric satisfaction scores (NPS, 1–5 ratings) tell us *how much* participants liked the programme.  
Open-text feedback tells us *why* — and crucially *what to change*.  
Without systematic text analysis, teams rely on cherry-picking quotes, missing systematic patterns.

### Methods used
1. **Frequency analysis** — most common words and bigrams in feedback  
2. **TF-IDF (Term Frequency–Inverse Document Frequency)** — words that are distinctive of each workshop's feedback, not just common in general  
3. **LDA Topic Modelling (Latent Dirichlet Allocation)** — unsupervised discovery of recurring themes across all open feedback  
4. **Sentiment-stratified analysis** — separating themes in positive vs. critical comments

### Why TF-IDF and LDA?
- **TF-IDF**: tells us which words *distinguish* one workshop from another — useful for comparing what made each experience unique  
- **LDA**: identifies hidden themes without pre-labelling — useful when we don't know in advance what participants will mention

### Data source
- `data/workshop_participants.csv` — column `survey_open_feedback` contains open-text responses  
- `data/reputation_testimonials.csv` — external stakeholder testimonials  
- `data/reputation_social_mentions.csv` — social media mentions

---

In [ ]:
# ─── Cell 1 · Imports ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from collections import Counter
import re
import warnings
warnings.filterwarnings("ignore")

# Scikit-learn: TF-IDF vectoriser and LDA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

DATA_DIR = Path("data")
print("Libraries loaded. scikit-learn available for TF-IDF and LDA.")

---
## Section 1 — Load and clean feedback data

In [ ]:
# ─── Cell 2 · Load feedback data ─────────────────────────────────────────────
participants = pd.read_csv(DATA_DIR / "workshop_participants.csv", parse_dates=["workshop_date"])
testimonials = pd.read_csv(DATA_DIR / "reputation_testimonials.csv")
social       = pd.read_csv(DATA_DIR / "reputation_social_mentions.csv")

# Extract non-null feedback responses
feedback_df = participants[["workshop_name", "modality", "payment_type",
                             "survey_overall", "survey_nps", "survey_open_feedback"]].dropna(
    subset=["survey_open_feedback"]).copy()

# Strip whitespace
feedback_df["survey_open_feedback"] = feedback_df["survey_open_feedback"].str.strip()

print(f"Total participant responses with open feedback: {len(feedback_df)} / {len(participants)}")
print(f"Response rate for open text: {len(feedback_df)/len(participants)*100:.1f}%")
print(f"\nUnique feedback strings: {feedback_df['survey_open_feedback'].nunique()}")
print("\nSample feedback texts:")
for text in feedback_df["survey_open_feedback"].unique():
    print(f"  - {text}")

In [ ]:
# ─── Cell 3 · Text preprocessing ─────────────────────────────────────────────
# We normalise the text before any analysis:
# - Lowercase everything
# - Remove punctuation (keep word characters and spaces)
# - Remove common English stop words (handled by sklearn)
# Note: Our feedback corpus is small (from synthetic data), so we keep all unique
# strings for analysis even if they repeat across participants.

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

feedback_df["clean_text"] = feedback_df["survey_open_feedback"].apply(clean_text)

# All texts (including repeated ones — they represent multiple participants)
all_texts = feedback_df["clean_text"].tolist()

# All unique message types (for pattern exploration)
unique_texts = list(feedback_df["clean_text"].unique())

print(f"Total feedback texts (incl. repeats): {len(all_texts)}")
print(f"Unique feedback messages             : {len(unique_texts)}")

---
## Section 2 — Word frequency analysis

**Why start with frequency?**  
Simple word counts are transparent, reproducible, and easy to explain to non-technical stakeholders.  
They reveal the vocabulary of appreciation and complaint most commonly used by participants.

In [ ]:
# ─── Cell 4 · Word frequency (unigrams and bigrams) ──────────────────────────
# We split all texts into words and count frequency.
# Stop words (a, the, and, ...) are excluded to focus on content words.
STOP_WORDS = {
    "the", "a", "an", "i", "was", "were", "is", "it", "but", "and", "or",
    "for", "to", "of", "in", "very", "could", "at", "by", "we", "had",
    "be", "been", "more", "would", "have", "with", "that", "this",
    "some", "my", "our", "are", "has", "too", "also", "so", "from",
    "on", "not", "out", "bit", "bit", "end"
}

all_words = []
for text in all_texts:
    words = text.split()
    all_words.extend([w for w in words if w not in STOP_WORDS and len(w) > 2])

word_freq = Counter(all_words).most_common(25)
words_df  = pd.DataFrame(word_freq, columns=["word", "count"])

print("Top 25 most frequent words in feedback:")
print(words_df.to_string(index=False))

In [ ]:
# ─── Cell 5 · Word frequency bar chart ───────────────────────────────────────
# Colour-coding: words associated with praise are green, critique are red.
PRAISE_WORDS   = {"great", "excellent", "practical", "good", "useful", "clear",
                  "well", "patient", "helpful", "best", "thoroughly"}
CRITIQUE_WORDS = {"outdated", "rushed", "confusing", "short", "slow", "issues",
                  "lacking", "small", "ran"}

bar_colors = []
for w in words_df["word"]:
    if w in PRAISE_WORDS:
        bar_colors.append("seagreen")
    elif w in CRITIQUE_WORDS:
        bar_colors.append("tomato")
    else:
        bar_colors.append("steelblue")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(words_df["word"], words_df["count"], color=bar_colors, edgecolor="white")
ax.set_xticklabels(words_df["word"], rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Frequency")
ax.set_title("Most Frequent Words in Open Feedback (green=praise, red=critique)", fontweight="bold")

from matplotlib.patches import Patch
legend = [Patch(color="seagreen", label="Praise"),
          Patch(color="tomato",   label="Critique"),
          Patch(color="steelblue",label="Neutral")]
ax.legend(handles=legend, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 6 · Bigram frequency ───────────────────────────────────────────────
# Bigrams (two-word sequences) preserve more meaning than single words.
# 'hands on' means something very specific that 'hands' and 'on' do not.
def get_bigrams(texts):
    bigrams = []
    for text in texts:
        words = [w for w in text.split() if w not in STOP_WORDS and len(w) > 2]
        bigrams.extend([(words[i], words[i+1]) for i in range(len(words)-1)])
    return bigrams

bigrams = get_bigrams(all_texts)
bigram_freq = Counter(bigrams).most_common(15)
bigrams_df  = pd.DataFrame([(" ".join(b), c) for b, c in bigram_freq],
                            columns=["bigram", "count"])

print("Top 15 bigrams:")
print(bigrams_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(bigrams_df["bigram"][::-1], bigrams_df["count"][::-1], color="mediumpurple", edgecolor="white")
ax.set_xlabel("Frequency")
ax.set_title("Most Frequent Bigrams in Open Feedback", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 3 — TF-IDF Analysis: distinctive terms per workshop

**What TF-IDF does**  
TF-IDF assigns each word a score that reflects:  
- **High score**: the word appears often in *this workshop's* feedback but rarely in *other workshops'* feedback → **distinctive**  
- **Low score**: the word appears everywhere (e.g. "workshop", "instructor") → not distinctive  

This is more informative than raw frequency because it separates the signal (what makes each workshop unique) from the noise (words everyone uses).

In [ ]:
# ─── Cell 7 · TF-IDF per workshop ─────────────────────────────────────────────
# We create one document per workshop by concatenating all feedback from that workshop.
# The TF-IDF vectoriser then computes scores across the 8-document corpus.
ws_feedback = (
    feedback_df
    .groupby("workshop_name")["clean_text"]
    .apply(lambda texts: " ".join(texts))
    .reset_index()
)
ws_feedback.columns = ["workshop", "combined_feedback"]

print(f"Workshops with feedback: {len(ws_feedback)}")
for _, row in ws_feedback.iterrows():
    word_count = len(row["combined_feedback"].split())
    print(f"  {row['workshop']:<40} {word_count} words")

In [ ]:
# ─── Cell 8 · Fit TF-IDF vectoriser ──────────────────────────────────────────
# max_features=50: keep the top 50 terms by importance across all workshops
# min_df=1: a term must appear in at least 1 document (workshop) to be included
# ngram_range=(1,2): include both single words and two-word phrases
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=60,
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf_vectorizer.fit_transform(ws_feedback["combined_feedback"])
feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=ws_feedback["workshop"],
    columns=feature_names
)

print(f"TF-IDF matrix shape: {tfidf_df.shape}  ({tfidf_df.shape[0]} workshops × {tfidf_df.shape[1]} terms)")

In [ ]:
# ─── Cell 9 · Top 5 TF-IDF terms per workshop ─────────────────────────────────
# For each workshop, we extract the 5 terms with the highest TF-IDF score.
# These are the words that MOST CHARACTERISE feedback about that specific workshop.
print("Top TF-IDF terms per workshop (most distinctive feedback keywords):\n")
for ws_name in tfidf_df.index:
    row = tfidf_df.loc[ws_name]
    top5 = row.nlargest(5)
    terms_str = "  |  ".join([f"{t} ({v:.3f})" for t, v in top5.items() if v > 0])
    print(f"  {ws_name}")
    print(f"    → {terms_str if terms_str else 'no distinctive terms'}")
    print()

In [ ]:
# ─── Cell 10 · TF-IDF heatmap across workshops ───────────────────────────────
# Show the top 20 globally important terms as a heatmap.
# Darker cells = the term is more distinctive for that workshop.
# This provides a visual signature of what each workshop's feedback is about.
top_terms_global = tfidf_df.max(axis=0).nlargest(20).index.tolist()
heatmap_tfidf = tfidf_df[top_terms_global]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_tfidf, annot=True, fmt=".2f", cmap="Blues",
            linewidths=0.5, ax=ax, cbar_kws={"label": "TF-IDF score"})
ax.set_title("TF-IDF Scores — Top 20 Distinctive Terms per Workshop", fontweight="bold")
ax.set_xlabel("Term")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=40, labelsize=8)
plt.tight_layout()
plt.show()

print("Reading the heatmap:")
print("  Dark blue = this term is very distinctive for that workshop's feedback.")
print("  White/light = this term does not characterise that workshop's feedback.")

---
## Section 4 — LDA Topic Modelling

**What LDA does**  
Latent Dirichlet Allocation treats each feedback text as a mixture of latent *topics*, where each topic is characterised by a distribution over words.  
Unlike TF-IDF (which is per-document), LDA identifies themes that recur *across all texts*.  

**Practical use**:  
If topic 2 clusters words like "hands-on", "exercises", "practical", "apply" — it signals that participants highly value applied learning. This is actionable: the team should ensure all workshops have enough hands-on exercises.

**Number of topics**  
With a small corpus (8–15 distinct feedback strings), we use **4 topics** — enough to capture meaningful themes without overfitting.

In [ ]:
# ─── Cell 11 · Fit LDA model ──────────────────────────────────────────────────
# We use a CountVectorizer (raw counts) for LDA — LDA expects count matrices, not TF-IDF scores.
# We analyse ALL individual feedback responses (not collapsed by workshop)
# so that LDA can discover themes across participants.

count_vectorizer = CountVectorizer(
    stop_words="english",
    max_features=40,
    ngram_range=(1, 2),
    min_df=1
)

count_matrix = count_vectorizer.fit_transform(all_texts)
count_feature_names = count_vectorizer.get_feature_names_out()

N_TOPICS = 4
lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    max_iter=200,
    learning_method="batch"
)
lda_model.fit(count_matrix)

print(f"LDA model fitted with {N_TOPICS} topics.")
print(f"Log-likelihood (higher = better fit): {lda_model.score(count_matrix):.2f}")
print(f"Perplexity (lower = better): {lda_model.perplexity(count_matrix):.2f}")

In [ ]:
# ─── Cell 12 · Print LDA topics and their top words ──────────────────────────
# For each topic, we print the 8 words with the highest probability.
# We then manually annotate each topic based on the word cluster.

TOPIC_LABELS = [
    "Applied learning & practice",   # Topic 0
    "Instructor quality",             # Topic 1
    "Content & materials",            # Topic 2
    "Organisation & logistics",       # Topic 3
]

print("=== LDA Topics (4 latent themes in participant feedback) ===\n")
for i, topic in enumerate(lda_model.components_):
    top_word_idx = topic.argsort()[-10:][::-1]
    top_words = [count_feature_names[idx] for idx in top_word_idx]
    label = TOPIC_LABELS[i] if i < len(TOPIC_LABELS) else f"Topic {i}"
    print(f"  Topic {i} — '{label}'")
    print(f"    Words: {', '.join(top_words)}")
    print()

print("Note: Topic labels above are manually assigned based on the dominant words.")
print("In a larger corpus, topic coherence scores (e.g., C_v) would validate the labelling.")

In [ ]:
# ─── Cell 13 · Topic distribution visualisation ──────────────────────────────
# Each feedback text is assigned a dominant topic. We then count how many
# responses belong to each topic.

topic_dist_all  = lda_model.transform(count_matrix)          # shape (n_texts, n_topics)
dominant_topics = np.argmax(topic_dist_all, axis=1)
feedback_df["dominant_topic"] = dominant_topics
feedback_df["dominant_topic_label"] = feedback_df["dominant_topic"].apply(
    lambda i: TOPIC_LABELS[i] if i < len(TOPIC_LABELS) else f"Topic {i}")

topic_counts = feedback_df["dominant_topic_label"].value_counts()
print("Number of feedback texts per dominant topic:")
print(topic_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
colors_lda = ["steelblue", "seagreen", "mediumpurple", "darkorange"]
topic_counts.plot(kind="bar", ax=axes[0], color=colors_lda[:len(topic_counts)], width=0.5)
axes[0].set_title("Feedback Texts per Topic", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=20)

# Pie chart
axes[1].pie(topic_counts, labels=topic_counts.index, autopct="%1.1f%%",
            colors=colors_lda[:len(topic_counts)], startangle=90)
axes[1].set_title("Topic Share of All Feedback", fontweight="bold")

plt.suptitle("LDA Topic Distribution Across Participant Feedback", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 14 · LDA topic word-weight bar charts ───────────────────────────────
# A grid of bar charts showing the word probabilities within each topic.
# This is the standard way to present LDA topics in academic papers and reports.

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes_flat = axes.flatten()

for i, (ax, topic) in enumerate(zip(axes_flat, lda_model.components_)):
    top_idx = topic.argsort()[-10:][::-1]
    top_words = [count_feature_names[idx] for idx in top_idx]
    top_weights = topic[top_idx]
    label = TOPIC_LABELS[i] if i < len(TOPIC_LABELS) else f"Topic {i}"

    ax.barh(top_words[::-1], top_weights[::-1], color=colors_lda[i], edgecolor="white")
    ax.set_title(f"Topic {i}: {label}", fontweight="bold", fontsize=10)
    ax.set_xlabel("Word weight")

plt.suptitle("LDA Topics — Top Words by Probability Weight", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## Section 5 — Topic distribution by workshop

**Why this matters**  
Not all workshops will generate the same mix of feedback themes.  
A workshop that triggers mostly "Organisation & logistics" comments may have scheduling issues.  
A workshop that triggers mostly "Applied learning" comments is successfully delivering practical value.  
This breakdown lets the team tailor improvements to specific workshops rather than applying blanket changes.

In [ ]:
# ─── Cell 15 · Topic distribution per workshop (stacked bar) ─────────────────
# We compute the average topic probability for each workshop's feedback.
feedback_df_topics = feedback_df.copy()
topic_probs = pd.DataFrame(topic_dist_all,
                            columns=[f"T{i}: {TOPIC_LABELS[i] if i<len(TOPIC_LABELS) else i}"
                                     for i in range(N_TOPICS)])
feedback_df_topics = pd.concat([feedback_df_topics.reset_index(drop=True), topic_probs], axis=1)

topic_col_names = topic_probs.columns.tolist()
ws_topic_dist = (
    feedback_df_topics
    .groupby("workshop_name")[topic_col_names]
    .mean()
)

# Shorten workshop names for display
ws_topic_dist.index = [n[:30] for n in ws_topic_dist.index]

fig, ax = plt.subplots(figsize=(13, 5))
ws_topic_dist.plot(kind="bar", stacked=True, ax=ax,
                   color=colors_lda[:N_TOPICS], edgecolor="white", linewidth=0.4)
ax.set_title("Topic Mix in Feedback per Workshop (normalised to 1.0)", fontweight="bold")
ax.set_ylabel("Mean topic probability")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
ax.legend(fontsize=8, loc="upper right", bbox_to_anchor=(1.35, 1))
plt.tight_layout()
plt.show()

---
## Section 6 — Sentiment-stratified analysis

**Approach**  
We split feedback texts into two groups based on NPS score:  
- **Promoters** (NPS 9–10): what do they say? → amplify these things  
- **Detractors/Passives** (NPS ≤ 8): what do they say? → fix these things  

This is the simplest form of *sentiment-stratified* qualitative analysis and is directly actionable.

In [ ]:
# ─── Cell 16 · Promoter vs detractor/passive feedback ────────────────────────
promoter_texts = feedback_df[feedback_df["survey_nps"] >= 9]["survey_open_feedback"].dropna()
passive_texts  = feedback_df[feedback_df["survey_nps"] <= 8]["survey_open_feedback"].dropna()

print(f"Promoter feedback texts (NPS 9–10): {len(promoter_texts)}")
for t in promoter_texts.unique():
    print(f"  ✓ {t}")

print(f"\nPassive/Detractor feedback texts (NPS ≤ 8): {len(passive_texts)}")
for t in passive_texts.unique():
    print(f"  ! {t}")

In [ ]:
# ─── Cell 17 · TF-IDF comparison: promoters vs passives/detractors ─────────────
# We create two documents (one for promoters, one for others) and run TF-IDF
# to find what distinguishes the language of each group.

if len(promoter_texts) > 0 and len(passive_texts) > 0:
    promoter_doc = " ".join(promoter_texts.apply(clean_text))
    passive_doc  = " ".join(passive_texts.apply(clean_text))

    tfidf_sentiment = TfidfVectorizer(stop_words="english", max_features=30, ngram_range=(1, 2))
    tfidf_s_matrix  = tfidf_sentiment.fit_transform([promoter_doc, passive_doc])
    s_features      = tfidf_sentiment.get_feature_names_out()
    s_df = pd.DataFrame(tfidf_s_matrix.toarray(), index=["Promoters", "Passives/Detractors"],
                        columns=s_features)

    # Top distinctive terms for each group
    top_promoter = s_df.loc["Promoters"].nlargest(8)
    top_passive  = s_df.loc["Passives/Detractors"].nlargest(8)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    top_promoter.plot(kind="barh", ax=axes[0], color="seagreen")
    axes[0].set_title("What Promoters say (TF-IDF)", fontweight="bold")
    axes[0].set_xlabel("TF-IDF score")

    top_passive.plot(kind="barh",  ax=axes[1], color="tomato")
    axes[1].set_title("What Passives/Detractors say (TF-IDF)", fontweight="bold")
    axes[1].set_xlabel("TF-IDF score")

    plt.suptitle("Distinctive Language: Promoters vs Passives/Detractors", fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient texts for sentiment-stratified TF-IDF (need both promoter and passive texts).")

---
## Section 7 — External reputation text analysis

Testimonials and social mentions complement participant feedback with the perspective of *external stakeholders* — partners, the public, municipal authorities. We analyse their language to understand how the programme is perceived beyond the workshop room.

In [ ]:
# ─── Cell 18 · Testimonial and social mention word analysis ──────────────────
all_external = (
    list(testimonials["testimonial_text"].dropna()) +
    list(social["text"].dropna())
)

print("All external texts (testimonials + social):")
for t in all_external:
    print(f"  - {t}")

external_clean = [clean_text(t) for t in all_external]
external_words = []
for text in external_clean:
    external_words.extend([w for w in text.split() if w not in STOP_WORDS and len(w) > 3])

ext_freq = Counter(external_words).most_common(20)
ext_df   = pd.DataFrame(ext_freq, columns=["word", "count"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(ext_df["word"], ext_df["count"], color="steelblue", edgecolor="white")
ax.set_xticklabels(ext_df["word"], rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Frequency")
ax.set_title("Top Words in External Testimonials & Social Mentions", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 8 — Actionable improvement recommendations

**Synthesising the text analysis into concrete actions**  
The cells below translate the NLP findings into structured, prioritised improvement recommendations.

In [ ]:
# ─── Cell 19 · Improvement action plan synthesis ─────────────────────────────
# We combine insights from:
#   - Word frequency (most common feedback terms)
#   - TF-IDF (workshop-distinctive themes)
#   - LDA topics (latent feedback themes)
#   - Promoter vs. passive language contrast
#   - Satisfaction dimension scores from Notebook 3
#
# This produces a structured, evidence-based improvement plan.

improvements = [
    {
        "Priority": "HIGH",
        "Theme": "Materials quality",
        "Evidence": "'materials' and 'outdated' appear in feedback; survey_materials has lowest mean score",
        "Action": "Review and update all handouts and slide decks before next iteration. Use open-source/recent examples.",
        "Impact": "Directly improves survey_materials score; reduces detractor language"
    },
    {
        "Priority": "HIGH",
        "Theme": "Hands-on time",
        "Evidence": "'hands-on', 'practical', 'apply', 'exercises' are top bigrams and LDA topic 0 words",
        "Action": "Increase exercise time by 20–30% per session. Prepare real-world case studies from Geneva NGO sector.",
        "Impact": "Improves knowledge gain (Kirkpatrick L2) and survey_content score"
    },
    {
        "Priority": "MEDIUM",
        "Theme": "Time management / pacing",
        "Evidence": "'ran out of time', 'rushed' and 'end' appear in passive/detractor feedback",
        "Action": "Introduce time-checks at mid-session. Design 10% content buffer. Practice run each new workshop.",
        "Impact": "Reduces time-related complaints; improves organisation score"
    },
    {
        "Priority": "MEDIUM",
        "Theme": "Small group Q&A",
        "Evidence": "'smaller group', 'Q&A', 'group' appear in passive texts",
        "Action": "For workshops > 25 participants, introduce breakout rooms (online) or table groups (onsite) for Q&A.",
        "Impact": "Improves instructor interaction satisfaction; reduces 'too large group' comments"
    },
    {
        "Priority": "LOW",
        "Theme": "Advanced content track",
        "Evidence": "External testimonial: 'bit more advanced content would be welcome for those who complete the basics'",
        "Action": "Design a Level 2 version of Python, Statistics, and Data Literacy for returning participants.",
        "Impact": "Retains alumni; expands programme reach; increases repeat NPS promoters"
    },
    {
        "Priority": "LOW",
        "Theme": "Online audio/platform quality",
        "Evidence": "'audio issues' in social mention; 'platform' survey dimension below 4.0 target",
        "Action": "Test audio setup 30min before each online session. Provide backup phone dial-in number.",
        "Impact": "Improves survey_platform score; reduces neutral/negative social mentions"
    },
]

imp_df = pd.DataFrame(improvements)
print("=" * 80)
print("  FEEDBACK-DRIVEN IMPROVEMENT ACTION PLAN — Priority Order")
print("=" * 80)
for _, row in imp_df.iterrows():
    print(f"\n  [{row['Priority']}] {row['Theme']}")
    print(f"  Evidence : {row['Evidence']}")
    print(f"  Action   : {row['Action']}")
    print(f"  Impact   : {row['Impact']}")
print("\n" + "=" * 80)

In [ ]:
# ─── Cell 20 · Action plan visualisation ─────────────────────────────────────
# A simple priority matrix (effort vs impact) for the improvement actions.
# This is the standard tool used in programme management to prioritise actions.

priority_matrix = pd.DataFrame([
    {"Action": "Update materials",        "Effort": 2, "Impact": 4, "Priority": "HIGH"},
    {"Action": "More hands-on exercises",  "Effort": 3, "Impact": 5, "Priority": "HIGH"},
    {"Action": "Better time management",   "Effort": 2, "Impact": 3, "Priority": "MEDIUM"},
    {"Action": "Small group Q&A",          "Effort": 3, "Impact": 3, "Priority": "MEDIUM"},
    {"Action": "Advanced track design",    "Effort": 5, "Impact": 4, "Priority": "LOW"},
    {"Action": "Fix audio/platform",       "Effort": 1, "Impact": 2, "Priority": "LOW"},
])

color_map_p = {"HIGH": "tomato", "MEDIUM": "darkorange", "LOW": "steelblue"}
colors_p = priority_matrix["Priority"].map(color_map_p)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(priority_matrix["Effort"], priority_matrix["Impact"],
           c=colors_p, s=200, zorder=5)

for _, row in priority_matrix.iterrows():
    ax.annotate(row["Action"], (row["Effort"], row["Impact"]),
                textcoords="offset points", xytext=(7, 3), fontsize=9)

ax.axvline(3, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(3, color="gray", linestyle="--", linewidth=0.8)

ax.text(0.8, 4.7, "Quick wins",    fontsize=9, color="gray", style="italic")
ax.text(3.3, 4.7, "Major projects",fontsize=9, color="gray", style="italic")
ax.text(0.8, 1.2, "Low priority",  fontsize=9, color="gray", style="italic")
ax.text(3.3, 1.2, "Re-evaluate",   fontsize=9, color="gray", style="italic")

ax.set_xlabel("Implementation Effort (1=low, 5=high)")
ax.set_ylabel("Expected Impact on Quality (1=low, 5=high)")
ax.set_title("Improvement Priority Matrix", fontweight="bold")
ax.set_xlim(0, 6)
ax.set_ylim(0, 6)

from matplotlib.patches import Patch
legend_p = [Patch(color="tomato", label="HIGH priority"),
             Patch(color="darkorange", label="MEDIUM priority"),
             Patch(color="steelblue", label="LOW priority")]
ax.legend(handles=legend_p, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 21 · Final summary: feedback analysis conclusions ──────────────────
print("═" * 70)
print("   FEEDBACK & TEXT ANALYSIS — SUMMARY")
print("═" * 70)
print(f"  Open feedback response rate          : {len(feedback_df)/len(participants)*100:.1f}%")
print(f"  Unique feedback themes identified    : {N_TOPICS} LDA topics")
print(f"  Dominant theme                       : {topic_counts.idxmax()}")
print(f"  Key improvement area (HIGH priority) : Materials quality + hands-on exercises")
print(f"  Quick win (LOW effort / MEDIUM impact): Fix audio/platform before online sessions")
print(f"  Promoter language highlights         : practical, instructor, clear, apply")
print(f"  Detractor language highlights        : outdated, rushed, time, group")
print()
print("  The text analysis converges on a clear message:")
print("  participants VALUE the practical, instructor-led format.")
print("  They want MORE hands-on time and FRESHER materials.")
print("  Logistics and pacing issues are fixable with low effort.")
print("═" * 70)